<div style="text-align: center;">

# Asset Impact: Request and Visualize Hazard and Impact Distributions

</div>

This notebook shows the end-to-end workflow for requesting asset-level climate impacts from the PhysRisk API and visualizing the distributions returned by the service.

The workflow is:

1. Load API configuration and credentials from the local environment.
2. Query the API for the hazard data sources available to the current API key.
3. Define an example real-estate asset with location and classification metadata.
4. Select the hazard indicators to assess.
5. Build and submit an asset impact request for selected scenarios and years.
6. Inspect the returned asset-impact records.
7. Plot the binned hazard intensity distribution and the resulting binned impact distribution.

The example focuses on `RiverineInundation` and the `flood_depth` indicator for a single `RealEstateAsset`, but the same request structure can be extended to multiple assets, hazards, scenarios, and years.

## 1. Imports

The notebook uses `requests` to call the API, `pandas` to inspect tabular response data, and `plotly` to visualize the binned distributions returned in the response.

In [1]:
import os

import pandas as pd
import plotly.graph_objects as go
import requests
from dotenv import load_dotenv

## 2. Load API Configuration

The API base URL and API key are read from `../.env`. The API key is sent in the `X-API-Key` header for each request.

Expected environment variables:

- `ALPHA_KLIMA_API_KEY`: API key used to authenticate API calls.
- `ALPHA_KLIMA_API_BASE_URL`: base URL of the Alpha-Klima API, for example the host that exposes `/api/get_available_sources` and `/api/get_asset_impact_clean`.

In [2]:
_ = load_dotenv("../.env")
API_KEY = os.environ.get("ALPHA_KLIMA_API_KEY")
API_BASE = os.environ.get("ALPHA_KLIMA_API_BASE_URL", "").rstrip("/")

## 3. Discover Available Hazard Sources

Before requesting impacts, query `/api/get_available_sources` to see which hazard datasets are available for the configured API key and use case.

The response is stored as `dict_available_sources`. It contains metadata by hazard family, including the models, indicators, scenarios, and years that can be requested. Use this information to avoid building an asset impact request with unsupported hazard or scenario combinations.

In [3]:
request = {
    "include_all": False,
    "use_case_id": "ECB_SCORES",
    "selected_hazards_list": [],
}
headers = {"X-API-Key": API_KEY, "Accept": "application/json"}

available_sources = requests.post(
    url=f"{API_BASE}/api/get_available_sources", json=request, headers=headers
)

dict_available_sources = available_sources.json()

In [4]:
pd.DataFrame(dict_available_sources["hazards"]["RiverineInundation"])

,flood_depth
scenarios,"[{'id': 'historical', 'years': [1985]}, {'id':..."
available_for_assets_type,"[PowerInfrastructureAsset, RealEstateAsset, In..."
indicator_display_name,Flood depth (TUDelft)


## 4. Define the Asset to Assess

The asset impact endpoint expects an `assets` payload with an `items` list. Each item represents one asset and includes the fields needed to locate the asset and select the relevant vulnerability model.

This example defines one industrial real-estate asset in Spain using latitude and longitude coordinates. In production workflows, this list would usually be generated from an asset register and may contain many assets in a single request.

### Example Asset Payload

The `asset_class` and `type` values are important because they determine which impact or vulnerability functions can be applied to the hazard intensity at the asset location.

In [5]:
asset = {"items": [
    {
        "id" : '0',
        "asset_name" : "Europe_1",
        "asset_class" : "RealEstateAsset",
        "type" : "Buildings/Industrial",
        "location" : "Europe",
        "country" : "ES",
        "latitude" : 41.3070,
        "longitude" : 2.0567,
    },
]}

asset_financial = {
    "asset value" : 25_000_000,
}

## 5. Select Hazard Indicators

`hazard_dictionary` defines the hazard scope for the asset impact calculation. Keys are hazard families and values are the indicators to request for each family.

The active example requests riverine flood depth:

- `RiverineInundation`: `flood_depth`

The commented entries show other possible hazard families and indicators. Only uncomment combinations that are available in `dict_available_sources` and supported for the selected `asset_class` and `type`.

In [6]:
hazard_dictionary ={
    "RiverineInundation": ["flood_depth"],
    # "Wind": ["wind_speed/3s"],
    # "CoastalInundation": ["flood_depth"],
    # "Drought": ["spi6", "cdd"],
    # "Landslide" : ["landslide_susceptability"],
    # "Subsidence": ["subsidence_susceptability"],
    # "WaterRisk": ["water_stress"],
    # "Fire" : ["fire_probability"],
    # "GroundShaking": ["pga_10pc50yr"],
    # "ChronicHeat": ["mean_degree_days/above/index", "mean_degree_days/above/32c", "mean_work_loss/high"]
}

## 6. Build the Asset Impact Request

`request_dict` is the payload sent to `/api/get_asset_impact_clean`. It combines the asset definition, the requested climate dimensions, and calculation settings.

Important fields:

- `assets`: the asset payload defined above.
- `use_case_id`: the API configuration to apply for this calculation.
- `years`: future horizons to request.
- `scenarios`: climate scenarios to request.
- `include_calc_details`: includes intermediate calculation outputs such as hazard distributions.
- `clean_response`: returns a simplified response structure for analysis.
- `calc_settings.hazard_scope_by_indicator`: passes the selected hazard indicators.
- `calc_settings.interpolate_years`: controls whether years outside the native dataset horizons should be interpolated.

This example requests `rcp4p5` for 2035 and 2085 without interpolation.

In [7]:
request_dict = {
    "assets":  asset,
    "include_asset_level": True,
    "include_measures": True,
    "include_calc_details": True, 
    "clean_response": True,
    "use_case_id": 'ECB_SCORES',
    "years": [2035, 2085],
    "scenarios": ["rcp4p5"],
    "calc_settings": {
        "hazard_scope_by_indicator": hazard_dictionary,
        "interpolate_years": False
    },
}

## 7. Execute the Asset Impact Request

The request is submitted to `/api/get_asset_impact_clean` using the same API-key authentication. The JSON response is stored as `dict_asset_impact`.

For each requested asset, the response contains an `impacts` list. Each impact record corresponds to a hazard, indicator, model, scenario, and year combination. With `include_calc_details=True`, each record can also include the binned hazard distribution used during the impact calculation.

In [8]:
headers = {"X-API-Key": API_KEY, "Accept": "application/json"}

asset_impact = requests.post(
    url=f"{API_BASE}/api/get_asset_impact_clean", json=request_dict, headers=headers
)

dict_asset_impact = asset_impact.json()

### Inspect One Asset Result

Display the first item in `asset_impacts` to understand the response structure before transforming it. This is the easiest way to identify the available metadata fields and confirm where the impact and calculation-detail distributions are stored.

In [9]:
dict_asset_impact["asset_impacts"][0]

{'asset_id': '0',
 'impacts': [{'key': {'hazard_type': 'RiverineInundation',
    'scenario_id': 'historical',
    'year': '1985'},
   'impact_type': 'damage',
   'hazard_indicator_id': 'flood_depth',
   'impact_distribution': {'bin_edges': [0.6837, 0.709, 0.7324, 0.7324],
    'probabilities': [0.006667, 0.002333, 0.001]},
   'impact_exceedance': {'values': [0.6837, 0.709, 0.7324, 0.7324],
    'exceed_probabilities': [0.01, 0.003333, 0.001, 0.0]},
   'impact_mean': 0.007056,
   'impact_std_deviation': 0.07023,
   'impact_semi_std_deviation': 0.06988,
   'calc_details': {'hazard_exceedance': {'values': [2.91, 3.06, 3.216, 3.216],
     'exceed_probabilities': [0.01, 0.003333, 0.001, 0.0]},
    'hazard_distribution': {'bin_edges': [2.91, 3.06, 3.216, 3.216],
     'probabilities': [0.006667, 0.002333, 0.001]},
    'vulnerability_distribution': {'intensity_bin_edges': [2.91,
      3.06,
      3.216,
      3.216],
     'impact_bin_edges': [0.6837, 0.709, 0.7324, 0.7324],
     'prob_matrix': [

## 8. Extract Impact Records

For this single-asset example, `dict_asset_impact['asset_impacts'][0]['impacts']` contains the impact records returned for the asset. The following cells convert those records to a DataFrame so they can be inspected, filtered, and plotted more easily.

In [10]:
asset_impacts = dict_asset_impact['asset_impacts'][0]['impacts']

## 9. Define a Plotting Helper

Both hazard and impact distributions are represented as binned probability distributions with two arrays:

- `bin_edges`: the boundaries of the bins.
- `probabilities`: the probability mass assigned to each bin.

Plotly bar charts need one x-value and one width per bar, so `compute_bar_params` converts bin edges into bin midpoints and bin widths while passing through the probabilities as bar heights.

In [11]:
def compute_bar_params(distribution):
    """
    Given a distribution dictionary with keys 'bin_edges' and 'probabilities',
    compute the midpoint for each bin, the width of the bin (difference between
    adjacent bin edges), and return also the list of probabilities.
    """
    edges = distribution["bin_edges"]
    probs = distribution["probabilities"]
    midpoints = []
    widths = []
    # There should be len(edges)-1 bins.
    for i in range(len(edges) - 1):
        low = edges[i]
        high = edges[i+1]
        midpoints.append( (low + high) / 2 )
        widths.append( high - low )
    return midpoints, widths, probs


## 10. Convert Impact Records to a DataFrame

The DataFrame view makes it easier to inspect the returned columns, compare records across scenarios and years, and select a specific record for visualization. Typical columns include metadata about the hazard calculation, the `impact_distribution`, and, when requested, `calc_details`.

In [12]:
df_asset_impacts = pd.DataFrame(asset_impacts)

df_asset_impacts

,key,impact_type,hazard_indicator_id,impact_distribution,impact_exceedance,impact_mean,impact_std_deviation,impact_semi_std_deviation,calc_details
0,"{'hazard_type': 'RiverineInundation', 'scenari...",damage,flood_depth,"{'bin_edges': [0.6837, 0.709, 0.7324, 0.7324],...","{'values': [0.6837, 0.709, 0.7324, 0.7324], 'e...",0.007056,0.07023,0.06988,"{'hazard_exceedance': {'values': [2.91, 3.06, ..."
1,"{'hazard_type': 'RiverineInundation', 'scenari...",damage,flood_depth,"{'bin_edges': [0.6837, 0.7011, 0.7264, 0.7517,...","{'values': [0.6837, 0.7011, 0.7264, 0.7517, 0....",0.015550,0.10370,0.10260,"{'hazard_exceedance': {'values': [2.91, 3.007,..."
2,"{'hazard_type': 'RiverineInundation', 'scenari...",damage,flood_depth,"{'bin_edges': [0.6837, 0.7208, 0.7583, 0.7927,...","{'values': [0.6837, 0.7208, 0.7583, 0.7927, 0....",0.070040,0.21360,0.20300,"{'hazard_exceedance': {'values': [2.91, 3.139,..."


## 11. Visualize a Hazard Intensity Distribution

This section selects one impact record and plots the hazard distribution found in `calc_details['hazard_distribution']`.

Interpret the chart as follows:

- The x-axis shows hazard intensity bins, such as flood-depth intervals.
- The y-axis shows the probability mass assigned to each bin.

In [13]:
intensity_bin_edges = df_asset_impacts['calc_details'][2]['hazard_distribution']['bin_edges']
probabilities = df_asset_impacts['calc_details'][2]['hazard_distribution']['probabilities']

dist = dict()
dist['bin_edges'] = intensity_bin_edges
dist['probabilities'] = probabilities

fig = go.Figure()

midpoints, widths, probs = compute_bar_params(dist)
fig.add_trace(
    go.Bar(
    x = midpoints,
    y = probs,
    width = widths,
    marker_color = "rgba(44, 160, 44, 0.6)"
    )
)

fig.update_layout(
    font = dict(family = "TimesNewRoman", size = 20),
    title = "Intensity distribution for rcp8p5, 2035 - RiverineInundation (TUDelft)",
    title_font_size = 30,
)

fig.update_layout(
    xaxis = dict(
        title = 'Flood depth [m/s]'
    ),
    yaxis = dict(
        title = 'Probability [%]'
    )
)
fig

## 12. Visualize an Impact Distribution

Each impact record also contains an `impact_distribution`, which represents the probability distribution of the calculated impact for the selected asset and hazard.

Interpret the chart as follows:

- The x-axis shows impact bins, such as damage or loss intervals.
- The y-axis shows the probability mass assigned to each impact bin.

This distribution can be used to compare risk across scenarios and years, inspect tail risk, or derive summary metrics such as expected impact and percentile impacts.

In [14]:
df = pd.DataFrame(df_asset_impacts)

In [15]:
df_to_plot = df["impact_distribution"][2]

df_to_plot

{'bin_edges': [0.6837, 0.7208, 0.7583, 0.7927, 0.8302, 0.8302],
 'probabilities': [0.06391, 0.02333, 0.006667, 0.002333, 0.001]}

### Render the Impact Distribution Plot

The plot below visualizes the selected `impact_distribution` using the same bin-midpoint and bin-width helper.

In [16]:
midpoints, widths, probs = compute_bar_params(df_to_plot)
trace = go.Bar(
        x = midpoints,
        y = probs,
        width = widths,
        marker_color = '#001845',
        opacity = 0.6
)

fig = go.Figure(data=trace)

fig.update_layout(
    font=dict(family='TimesNewRoman', size = 18),
    title = "<b>Impact Distribution for Scenario rcp8p5 - 2035 (RiverineInundation - TUDelft)</b>",
    title_font_size = 20,
    xaxis_title = "Impact [damage]",
    yaxis_title = "Probability",
    barmode = "overlay",
    bargap = 0.0
)

fig.show()